# โครงการแรกของฉัน: ใช้รูปจากมือถือ ตรวจชิ้นงานของโรงงานตัวเอง

ไม่ต้องมีกล้อง Basler  ใช้**รูปถ่ายจากมือถือ** ก็ทดลองแนวคิดได้ครบ ก่อนตัดสินใจซื้ออุปกรณ์

**4 ขั้น**
1. กรอกข้อมูลโครงการ (ข้อ 1)
2. ถ่ายรูป ชิ้นงานดี 1 รูป + ชิ้นงานที่จะตรวจหลายรูป ใส่โฟลเดอร์ `images/my_project/`
3. เลือก 1 ใน 3 วิธี: **เทียบภาพอ้างอิง / แผนที่โซน / นับชิ้นงาน**
4. รันทุกรูป ได้ตาราง OK/NG และสรุปโครงการ 1 หน้า ไปคุยกับหัวหน้าได้

**ถ่ายรูปอย่างไรให้ได้ผล**
- วางมือถือบน**ขาตั้งหรือกองหนังสือ** ระยะเดิมทุกรูป (สำคัญที่สุด)
- พื้นหลังเรียบ สีตัดกับชิ้นงาน  ไม่ใช้แฟลช  แสงเดิมทุกรูป
- ตั้งชื่อรูปชิ้นงานดีว่า `good.jpg`  รูปอื่นชื่ออะไรก็ได้

ถ้ายังไม่มีรูป notebook จะใช้**รูปตัวอย่าง**ที่จำลองการถ่ายด้วยมือถือ (เอียงนิด ระยะต่างนิด) ให้ลองก่อน

In [ ]:
import sys, os, glob, csv
sys.path.insert(0, os.path.abspath('../scripts'))
import cv2
import numpy as np
from vision_camera import show, img_path, put_label
print('พร้อม')

## ข้อ 1: กรอกข้อมูลโครงการ
แก้ข้อความในเครื่องหมาย ' ' ให้เป็นของโรงงานคุณ

In [ ]:
PROJECT = dict(
    name     = 'ตรวจถาดบรรจุชิ้นส่วน',            # ชื่อโครงการ
    product  = 'ถาดใส่แหวนโลหะ 6 ช่อง',           # ชิ้นงานคืออะไร
    defect   = 'มีสิ่งแปลกปลอม หรือแหวนหาย',      # ของเสียที่ต้องการจับ
    now      = 'พนักงานดูด้วยตา 1 คน ต่อกะ',       # ตอนนี้ตรวจอย่างไร
    goal     = 'ไม่ให้ของเสียหลุดถึงลูกค้า',        # เป้าหมาย
    method   = 'reference',                      # 'reference' = เทียบภาพอ้างอิง, 'zones' = แผนที่โซน, 'count' = นับชิ้นงาน
)
print('โครงการ:', PROJECT['name'], ' วิธี:', PROJECT['method'])

## ข้อ 2: โหลดรูป
ถ้ามีโฟลเดอร์ `images/my_project/` ที่มี `good.jpg` จะใช้รูปของคุณ  ถ้าไม่มีจะสร้างรูปตัวอย่างให้

In [ ]:
MY_DIR = img_path('my_project')

def make_demo_photos():
    # จำลองรูปจากมือถือ: เอียง เลื่อน ย่อขยาย และแสงต่างกันนิดหน่อย
    rng = np.random.default_rng(3)
    names = ['tray_reference.png', 'tray_01_clean.png', 'tray_02_screw.png',
             'tray_03_hair.png', 'tray_04_missing.png', 'tray_05_plastic.png']
    photos = {}
    for k, n in enumerate(names):
        img = cv2.imread(img_path('industrial/' + n))
        h, w = img.shape[:2]
        if k == 0:
            photos['good.jpg'] = img
            continue
        ang = rng.uniform(-4, 4); sc = rng.uniform(0.95, 1.05)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
        M[:, 2] += rng.uniform(-15, 15, 2)
        warped = cv2.warpAffine(img, M, (w, h), borderValue=(70, 72, 75))
        warped = np.clip(warped.astype(np.float32) * rng.uniform(0.9, 1.1), 0, 255).astype(np.uint8)
        photos['photo_' + n.split('_', 1)[1].replace('.png', '.jpg')] = warped
    return photos

if os.path.exists(os.path.join(MY_DIR, 'good.jpg')):
    photos = {os.path.basename(p): cv2.imread(p)
              for p in sorted(glob.glob(os.path.join(MY_DIR, '*')))
              if p.lower().endswith(('.jpg', '.jpeg', '.png'))}
    print('ใช้รูปของคุณ', len(photos), 'รูป จาก', MY_DIR)
else:
    photos = make_demo_photos()
    print('ยังไม่มีโฟลเดอร์ my_project  ใช้รูปตัวอย่าง', len(photos), 'รูป')

# ย่อทุกรูปให้กว้าง 800 พิกเซล (รูปมือถือใหญ่มาก ประมวลผลช้า)
def resize800(img):
    s = 800 / img.shape[1]
    return cv2.resize(img, (800, int(img.shape[0] * s)))
photos = {k: resize800(v) for k, v in photos.items()}
good = photos.pop('good.jpg')
show(good, *list(photos.values())[:2], titles=['good.jpg (อ้างอิง)'] + list(photos)[:2], size=4)

## ข้อ 3: จัดตำแหน่งรูปให้ตรงกับรูปอ้างอิง
ถ่ายด้วยมือถือ แต่ละรูปจะเอียงและเลื่อนไม่เท่ากัน  ขั้นนี้ให้คอมพิวเตอร์**ค่อยๆ หมุน เลื่อน ย่อขยาย**รูป จนทับกับรูปอ้างอิงได้พอดีที่สุด (วิธี ECC)
เป็นขั้นที่ทำให้วิธีเทียบภาพอ้างอิงใช้ได้กับรูปจากมือถือ  ส่วนขอบรูปที่หายไปจากการหมุนจะไม่ถูกนับเป็นความต่าง

In [ ]:
g_good = cv2.GaussianBlur(cv2.cvtColor(good, cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)

def align_to_good(img):
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)
    warp = np.eye(2, 3, dtype=np.float32)
    try:
        score, warp = cv2.findTransformECC(g_good, g, warp, cv2.MOTION_AFFINE,
                                           (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 200, 1e-5), None, 5)
    except cv2.error:
        return img, np.full(good.shape[:2], 255, np.uint8), 0.0
    h, w = good.shape[:2]
    flags = cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP
    out = cv2.warpAffine(img, warp, (w, h), flags=flags, borderMode=cv2.BORDER_REPLICATE)
    valid = cv2.warpAffine(np.full(img.shape[:2], 255, np.uint8), warp, (w, h), flags=flags, borderValue=0)
    valid = cv2.erode(valid, np.ones((15, 15), np.uint8))          # ตัดขอบที่เกิดจากการหมุนทิ้ง
    return out, valid, score

aligned, valid_mask = {}, {}
for name, img in photos.items():
    aligned[name], valid_mask[name], score = align_to_good(img)
    print(f'{name:28s} ความเหมือนหลังจัดตำแหน่ง {score:.2f}  ' + ('จัดตำแหน่งได้' if score > 0.6 else 'จัดตำแหน่งไม่ดี: ถ่ายใหม่ให้มุมและระยะใกล้เคียงรูปอ้างอิง'))
first = list(photos)[0]
show(photos[first], aligned[first], good, titles=['ก่อนจัดตำแหน่ง', 'หลังจัดตำแหน่ง', 'รูปอ้างอิง'], size=4)

## ข้อ 4: ตรวจด้วยวิธีที่เลือก
ปรับค่าใน `SETTINGS` ให้เหมาะกับชิ้นงานของคุณ แล้วรันใหม่

| วิธี | ใช้เมื่อ | ค่าที่ปรับ |
|---|---|---|
| `reference` | หาอะไรก็ได้ที่ต่างจากชิ้นงานดี (สิ่งแปลกปลอม ชิ้นหาย) | `diff_threshold` ต่ำ = ไวขึ้น  `min_area` = ขนาดเล็กสุดที่สนใจ |
| `zones` | ตรวจทีละส่วน มีไหม / สว่างพอไหม | `ZONES` กรอบของแต่ละส่วน (ดูพิกัดจากรูปที่มีเส้นตาราง) |
| `count` | นับชิ้นงานที่ตัดกับพื้นหลัง | `expect` จำนวนที่ต้องมี  `invert` ถ้าชิ้นงานเข้มกว่าพื้น |

In [ ]:
SETTINGS = dict(diff_threshold=25, min_area=150, expect=6, invert=False, count_min_area=500)

# สำหรับวิธี zones: กรอบ (x, y, w, h) บนรูปขนาดกว้าง 800  ดูพิกัดจากรูปเส้นตารางด้านล่าง
ZONES = {'slot1': (95, 95, 70, 70), 'slot2': (345, 95, 70, 70), 'slot3': (595, 95, 70, 70)}

def show_grid(img, step=50):
    g = img.copy()
    for x in range(0, g.shape[1], step):
        cv2.line(g, (x, 0), (x, g.shape[0]), (0, 255, 255), 1)
        put_label(g, str(x), org=(x + 2, 14), scale=0.4)
    for y in range(0, g.shape[0], step):
        cv2.line(g, (0, y), (g.shape[1], y), (0, 255, 255), 1)
        put_label(g, str(y), org=(2, y + 14), scale=0.4)
    for name, (x, y, w, h) in ZONES.items():
        cv2.rectangle(g, (x, y), (x + w, y + h), (255, 0, 255), 2)
    show(g, titles=['รูปอ้างอิงพร้อมพิกัด (ใช้กำหนด ZONES)'], size=8)

def check_reference(img, valid):
    g1 = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (7, 7), 0)
    g0 = cv2.GaussianBlur(cv2.cvtColor(good, cv2.COLOR_BGR2GRAY), (7, 7), 0)
    _, m = cv2.threshold(cv2.absdiff(g1, g0), SETTINGS['diff_threshold'], 255, cv2.THRESH_BINARY)
    m = cv2.bitwise_and(m, valid)                                   # ไม่นับขอบรูป
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.dilate(m, np.ones((7, 7), np.uint8))
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bad = [c for c in cs if cv2.contourArea(c) > SETTINGS['min_area']]
    out = img.copy()
    for c in bad:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(out, (x, y), (x + w, y + h), (0, 0, 255), 3)
    return ('OK' if not bad else 'NG'), f'จุดที่ต่าง {len(bad)}', out

def check_zones(img, valid):
    out = img.copy(); bad = []
    g_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); g_ref = cv2.cvtColor(good, cv2.COLOR_BGR2GRAY)
    for name, (x, y, w, h) in ZONES.items():
        a = g_img[y:y + h, x:x + w].astype(float); b = g_ref[y:y + h, x:x + w].astype(float)
        same = abs(a.mean() - b.mean()) < 25 and abs(a.std() - b.std()) < 25
        cv2.rectangle(out, (x, y), (x + w, y + h), (0, 200, 0) if same else (0, 0, 255), 3)
        if not same:
            bad.append(name)
    return ('OK' if not bad else 'NG'), ('ส่วนที่ผิด ' + ', '.join(bad)) if bad else 'ครบทุกโซน', out

def check_count(img, valid):
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    flag = cv2.THRESH_BINARY_INV if SETTINGS['invert'] else cv2.THRESH_BINARY
    _, bw = cv2.threshold(gray, 0, 255, flag + cv2.THRESH_OTSU)
    cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    parts = [c for c in cs if cv2.contourArea(c) > SETTINGS['count_min_area']]
    out = img.copy(); cv2.drawContours(out, parts, -1, (0, 200, 0), 3)
    n = len(parts)
    return ('OK' if n == SETTINGS['expect'] else 'NG'), f'พบ {n} ชิ้น ต้องมี {SETTINGS[chr(101) + chr(120) + chr(112) + chr(101) + chr(99) + chr(116)]}', out

CHECK = {'reference': check_reference, 'zones': check_zones, 'count': check_count}[PROJECT['method']]
if PROJECT['method'] == 'zones':
    show_grid(good)

## ข้อ 5: ตรวจทุกรูป และบันทึกผล

In [ ]:
results = []
for name, img in aligned.items():
    verdict, detail, out = CHECK(img, valid_mask[name])
    results.append((name, verdict, detail))
    print(f'{name:28s} {verdict}   {detail}')
    show(out, titles=[f'{name}  {verdict}'], size=5)

with open('my_project_results.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f); w.writerow(['รูป', 'ผล', 'รายละเอียด']); w.writerows(results)
print('บันทึกผลที่ notebooks/my_project_results.csv')

## ข้อ 6: สรุปโครงการ 1 หน้า
คัดลอกข้อความที่ได้ไปใส่อีเมลหรือรายงานถึงหัวหน้าได้เลย  ช่อง **ผลที่ถูกต้อง** ให้คุณตรวจด้วยตาว่าโปรแกรมตัดสินถูกกี่รูป

In [ ]:
n_ng = sum(r[1] == 'NG' for r in results)
lines = [
    'สรุปโครงการ: ' + PROJECT['name'],
    '',
    'ชิ้นงาน: ' + PROJECT['product'],
    'ของเสียที่ต้องการจับ: ' + PROJECT['defect'],
    'วิธีตรวจปัจจุบัน: ' + PROJECT['now'],
    'เป้าหมาย: ' + PROJECT['goal'],
    'วิธีที่ทดลอง: ' + {'reference': 'เทียบภาพอ้างอิง', 'zones': 'แผนที่โซน', 'count': 'นับชิ้นงาน'}[PROJECT['method']],
    f'ผลทดลอง: {len(results)} รูป  ตัดสิน NG {n_ng} รูป  OK {len(results) - n_ng} รูป',
    'ผลที่ถูกต้องเมื่อตรวจด้วยตา: ___ / ' + str(len(results)) + ' รูป',
    '',
    'ขั้นต่อไป:',
    '1. ถ่ายรูปเพิ่มให้ครบทุกแบบของเสีย อย่างน้อย 20 รูป',
    '2. ทำขาตั้งยึดมือถือหรือกล้อง ให้ระยะคงที่ และคุมแสง',
    '3. ถ้าแม่นพอ ทดลองกล้องอุตสาหกรรม + จิ๊กที่หน้างาน 1 จุด',
]
print(chr(10).join(lines))

## ติดตรงไหน ใช้ AI ช่วย
เปิดไฟล์ `docs/AI_HELP.md` ในโฟลเดอร์นี้ มีตัวอย่างคำถามที่ใช้กับ ChatGPT, Claude หรือ Copilot ได้ทันที เช่น

> ใน notebook 10 ข้อ 4 รูปของฉันชิ้นงานสีเข้มบนพื้นขาว นับได้ 0 ชิ้น ช่วยบอกว่าต้องแก้ค่าอะไร และอธิบายทีละบรรทัด

**ข้อควรระวัง:** รูปชิ้นงานของโรงงานอาจเป็นความลับ อ่านนโยบายบริษัทก่อนอัปโหลดรูปขึ้นบริการ AI ภายนอก